# glassbox — Phase 3, TinyStories

The first run on a corpus that does not fit in memory, with a model large
enough to write coherent English. Roughly **11M parameters**, a 4,096-token BPE
vocabulary, and a 256-token context.

Two stages. **Preparation** downloads 1.9 GB, learns the merges and writes token
files — run once, and it lands in Drive so a disconnect does not repeat it.
**Training** memory-maps those files, so nothing is tokenised while the GPU is
waiting.

If the session drops mid-training, set `RESUME = True` and rerun the training
cell. It continues from `last.pt` with the optimizer moments intact.

## 1 · Hardware

In [ ]:
import subprocess

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                      "--format=csv,noheader"],
                     capture_output=True, text=True).stdout or "no GPU visible")

import torch
if torch.cuda.is_available():
    major, minor = torch.cuda.get_device_capability()
    print(f"device       {torch.cuda.get_device_name(0)}")
    print(f"capability   {major}.{minor}  "
          f"({'Ampere or newer' if major >= 8 else 'pre-Ampere'})")
    print(f"bf16 native  {major >= 8}")
else:
    print("No GPU. Runtime > Change runtime type > GPU.")

## 2 · Clone, install, mount

In [ ]:
import os, subprocess

ROOT = "/content/glassbox"
if os.path.isdir(ROOT):
    print(subprocess.run(["git", "-C", ROOT, "pull", "--ff-only"],
                         capture_output=True, text=True).stdout)
else:
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/udit-rawat/glassbox.git", ROOT], check=True)

os.chdir(ROOT)
subprocess.run(["pip", "install", "-q", "-e", ".", "--no-deps"], check=True)

from google.colab import drive
drive.mount("/content/drive")

DRIVE_DATA = "/content/drive/MyDrive/glassbox/data/tinystories"
LOCAL_DATA = "/content/data/tinystories"
OUT_DIR    = "/content/drive/MyDrive/glassbox/tinystories"
os.makedirs(OUT_DIR, exist_ok=True)

from glassbox.device import get_device
from glassbox.training.precision import select_precision
p = select_precision(get_device(), enabled=True)
print(f"precision   {p.describe()}   scaler {p.use_scaler}")

## 3 · Prepare the corpus

Downloads TinyStories, learns 3,840 merges from a 40 MB sample, and encodes the
chosen slice into flat `uint16` files.

Merges are learned from a sample rather than the whole corpus on purpose:
TinyStories uses a deliberately small vocabulary, so 40 MB already contains
almost every word, and more text would multiply the cost for merges that come
out nearly identical.

**Skip this cell entirely on a rerun** — it detects the files in Drive and
returns immediately.

In [ ]:
TRAIN_MB   = 500     # of the 1.9 GB corpus
VOCAB_SIZE = 4096
BPE_MB     = 40

prepared = all(os.path.exists(f"{DRIVE_DATA}/{f}")
               for f in ("train.bin", "val.bin", "tokenizer.json"))

if prepared:
    print(f"already prepared in {DRIVE_DATA} — skipping")
else:
    cmd = ["python", "-u", "scripts/prepare_tinystories.py",
           "--data-dir", "/content/data/raw",
           "--out-dir", DRIVE_DATA,
           "--train-mb", str(TRAIN_MB),
           "--vocab-size", str(VOCAB_SIZE),
           "--bpe-mb", str(BPE_MB)]
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end="")
    proc.wait()

## 4 · Copy the token files to local disk

Drive is a network mount. Memory-mapping across it means every batch waits on a
round trip, which starves the GPU exactly as tokenising in Python would. The
files stay in Drive as the durable copy; training reads a local one.

In [ ]:
import shutil, time

os.makedirs(LOCAL_DATA, exist_ok=True)
for f in ("train.bin", "val.bin", "tokenizer.json"):
    dst = f"{LOCAL_DATA}/{f}"
    if not os.path.exists(dst):
        t0 = time.time()
        shutil.copy(f"{DRIVE_DATA}/{f}", dst)
        print(f"{f:<16} {os.path.getsize(dst)/1e6:>8.1f} MB  {time.time()-t0:5.1f}s")
    else:
        print(f"{f:<16} {os.path.getsize(dst)/1e6:>8.1f} MB  (already local)")

## 5 · Sanity check

In [ ]:
print(subprocess.run(["python", "-m", "pytest", "tests/", "-q", "--no-header"],
                     capture_output=True, text=True).stdout[-800:])

## 6 · Train

Mixed precision is on here, unlike the Phase 1 restore — there is no old number
to reproduce, so speed wins. The header prints how many tokens the run will
consume against how many exist; above 1.0 epochs the model is seeing the corpus
more than once.

Watch the first two evaluations. Loss should start near `ln(4096) ≈ 8.3` and
fall quickly. If it sits flat, stop and check rather than waiting hours.

In [ ]:
MAX_ITERS  = 20000
BATCH_SIZE = 32
GRAD_ACCUM = 1
LR         = 6e-4
RESUME     = False   # True after a disconnect

cmd = ["python", "-u", "scripts/train_tinystories.py",
       "--data-dir", LOCAL_DATA,
       "--out-dir", OUT_DIR,
       "--max-iters", str(MAX_ITERS),
       "--batch-size", str(BATCH_SIZE),
       "--grad-accum", str(GRAD_ACCUM),
       "--lr", str(LR),
       "--schedule", "cosine",
       "--eval-interval", "500",
       "--sample-tokens", "300"]
if RESUME:
    cmd.append("--resume")

print(" ".join(cmd), "\n")
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end="")
proc.wait()

## 7 · Sample

The real test of Phase 3: does it write English? Nucleus sampling rather than
top-k, because the candidate set should widen where the story could go several
ways and narrow mid-word.

In [ ]:
for prompt in ["Once upon a time, there was a little",
               "Tom and Sara went to the park. They saw",
               "The dog was very sad because"]:
    print("=" * 70)
    print(subprocess.run([
        "python", "scripts/sample.py",
        "--checkpoint", f"{OUT_DIR}/best.pt",
        "--prompt", prompt,
        "--tokens", "200",
        "--temperature", "0.8",
        "--top-p", "0.9",
    ], capture_output=True, text=True).stdout)

## 8 · What came back

In [ ]:
import math, torch
from pathlib import Path

for f in sorted(Path(OUT_DIR).iterdir()):
    print(f"{f.name:<20} {f.stat().st_size / 1e6:>8.1f} MB")

ckpt = torch.load(f"{OUT_DIR}/best.pt", map_location="cpu", weights_only=False)
cfg = ckpt["model_config"]
loss = ckpt["val_loss"]
print(f"\niter        {ckpt['iter']:,}")
print(f"val loss    {loss:.4f}")
# Perplexity reads as an effective number of choices per token. Starting from
# ln(vocab), anything near single digits means the model has real structure.
print(f"perplexity  {math.exp(loss):.1f}   (uniform would be {cfg.vocab_size})")
print(f"arch        {cfg.norm} / {cfg.activation} / {cfg.pos_encoding} / kv={cfg.n_kv_heads}")
print(f"tokenizer   {ckpt['tokenizer']['kind']}, {len(ckpt['tokenizer']['merges'])} merges")